<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_12_modules_stdlib/note_lesson_12_modules_stdlib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 12 — Модулі та стандартна бібліотека

> Сценарій: ресторан веде записи замовлень із мітками часу (`timestamp`). Власнику потрібна аналітика по роках, місяцях, днях тижня, годинах — усе це вимагає роботи з датою й часом, а Python сам це не вміє: потрібні модулі `datetime`, `calendar`, `locale`, `collections`. Цей урок — про те, як Python знаходить і завантажує код, і як користуватись стандартною бібліотекою («batteries included»), а не лише зовнішніми `pip install`-пакетами.

## 🔁 RETRIEVE — пригадай попередні уроки (без підглядання)

1. Що таке `NamedTuple` і чим він відрізняється від звичайного `dict`?
2. Що поверне `d.get("x", 0)`, якщо ключа `"x"` немає?
3. Чим list comprehension `[x for x in items if cond]` відрізняється від звичайного `for`-циклу з `if`?

## 📖 CONCEPT

### 1. Навіщо існують модулі?

**Будь-який `.py`-файл — це модуль.** Без модулів увесь код довелося б тримати в одному файлі — конфлікти імен, неможливо повторно використати код в іншому проєкті.

```
my_project/
    main.py        ← модуль
    utils.py       ← модуль
```

Модуль — це окремий **простір імен** (namespace). Дві функції з однаковою назвою в різних модулях не конфліктують:

```python
# xml_reader.py
def parse(data): ...

# json_reader.py
def parse(data): ...   # інша функція — інший простір імен

import xml_reader, json_reader
xml_reader.parse(...)   # зрозуміло, яка саме
json_reader.parse(...)
```

In [ ]:
# __name__ — як Python розрізняє "запускається напряму" vs "імпортується"

def add(a, b):
    return a + b

# Цей блок виконується ТІЛЬКИ при прямому запуску файлу, не при import
if __name__ == "__main__":
    print(f"Тест: add(2, 3) = {add(2, 3)}")

# У ноутбуці __name__ завжди '__main__' — ми запускаємо напряму
print(f"__name__ = {__name__!r}")
# Якби цей код лежав у файлі my_math.py і хтось написав `import my_math`,
# __name__ усередині my_math.py дорівнював би 'my_math', і блок if не виконався б

### 2. Що відбувається при `import`?

```
import math
       │
       ▼
1. SEARCH   — шукаємо math.py за sys.path
2. COMPILE  — компілюємо в байткод
3. EXECUTE  — виконуємо файл зверху вниз
4. CACHE    — зберігаємо в sys.modules['math']
```

Python **ніколи** не виконує файл модуля двічі — повторний `import math` бере вже готовий об'єкт із кешу `sys.modules`.

In [ ]:
import sys
import random

print(f"'random' у sys.modules: {'random' in sys.modules}")
print(f"Де фізично живе random: {sys.modules['random'].__file__}")

# Повторний import — не перечитує файл, бере з кешу
import random
print(f"Той самий об'єкт після повторного import: {id(sys.modules['random']) == id(random)}")
print(f"Всього модулів у кеші зараз: {len(sys.modules)}")

### 3. `sys.path` і небезпека «затінення» (shadowing)

Python шукає модуль по директоріях `sys.path` у порядку: поточна папка → `PYTHONPATH` → стандартна бібліотека → `site-packages`.

**Класична помилка:** якщо назвати свій файл `random.py` або `math.py`, Python знайде **твій** файл першим — і стандартний модуль стане недоступним.

In [ ]:
import sys

print("Перші кілька шляхів пошуку модулів:")
for i, path in enumerate(sys.path[:4], 1):
    print(f"  {i}. {path or '(поточна папка)'}")

# Діагностика: якщо random.__file__ вказує на твій проєкт, а не на stdlib —
# у тебе є файл random.py, який "затінив" стандартний модуль
import random
print(f"\nrandom.__file__ = {random.__file__}")
assert "site-packages" not in random.__file__ or True  # діагностичний вивід, не завжди site-packages
print("Якщо шлях веде НЕ до стандартної бібліотеки Python — це затінення")

### 4. Три стилі імпорту

| Стиль | Синтаксис | Коли |
|---|---|---|
| Модуль цілком | `import math` | багато функцій, потрібна ясність (`math.pi`) |
| Конкретні імена | `from math import pi, sqrt` | кілька функцій, часто використовуються |
| З псевдонімом | `from math import pi as PI` | довгі назви, уникнення конфліктів |

Правило: перед тим як шукати пакет на PyPI — перевір, чи немає потрібного в стандартній бібліотеці (`datetime`, `calendar`, `collections`, `re`, `json`, `math`, `os`...).

In [ ]:
import math
from math import pi, sqrt

print(f"math.pi  = {math.pi:.4f}")
print(f"pi       = {pi:.4f}")
print(f"Однакове значення: {math.pi == pi}")
print()

# Wildcard import (from math import *) — НЕ використовуємо:
# забруднює простір імен, незрозуміло звідки взялось ім'я
print("Правило: явний імпорт (import math / from math import X) — завжди,")
print("wildcard (from math import *) — ніколи.")

### 5. `datetime` — дата й час одним об'єктом

Тепер до самого сценарію уроку: аналітика ресторану по датах. `datetime` — стандартний модуль, що зберігає дату **і** час разом:

```
datetime(2024, 7, 20, 19, 45, 30)
          рік  міс день год  хв  сек
```

In [ ]:
from datetime import datetime

dt = datetime(2024, 7, 20, 19, 45, 30)

print(f"Об'єкт: {dt}")
print(f"  .year    = {dt.year}")
print(f"  .month   = {dt.month}")
print(f"  .day     = {dt.day}")
print(f"  .hour    = {dt.hour}")
print(f"  .minute  = {dt.minute}")
print(f"  .weekday() = {dt.weekday()}   (0=Пн, 6=Нд)")

now = datetime.now()
print(f"\ndatetime.now() = {now}")

### 6. `strftime` — datetime у рядок

`strftime` = **str**ing **f**rom **time**. Найкорисніші коди: `%Y` рік, `%m` місяць, `%d` день, `%H` година, `%A`/`%a` день тижня повний/короткий, `%B`/`%b` місяць повний/короткий.

In [ ]:
dt = datetime(2024, 7, 20, 19, 45, 30)

formats = [
    ("%Y-%m-%d", "ISO дата"),
    ("%d.%m.%Y", "дата по-українськи"),
    ("%A", "день тижня повний"),
    ("%B", "місяць повний"),
    ("%Y-%m", "рік-місяць (зручно для групування)"),
]

for fmt, desc in formats:
    print(f"  {fmt:<12} -> {dt.strftime(fmt):<20} ({desc})")

### 7. Порівняння `datetime`

`datetime` можна порівнювати як числа: `<`, `>`, `==`. Це дає простий спосіб перевірити, чи дата потрапляє у проміжок.

In [ ]:
dt1 = datetime(2024, 1, 15, 10, 0)
dt2 = datetime(2024, 8, 20, 18, 30)

print(f"dt1 < dt2  -> {dt1 < dt2}")
print(f"dt1.year == 2024 -> {dt1.year == 2024}")

start, end = datetime(2024, 6, 1), datetime(2024, 9, 1)
print(f"\nЧи dt2 між {start.date()} і {end.date()}? -> {start <= dt2 <= end}")
assert (start <= dt2 <= end) is True

### 8. Дані: генеруємо замовлення з мітками часу

Невеликий синтетичний датасет — 2000 замовлень за 2 роки (2023–2024), кожне зі своїм `timestamp`.

In [ ]:
import pandas as pd
import random
from datetime import timedelta

random.seed(42)

start, end = datetime(2023, 1, 1), datetime(2025, 1, 1)
delta = end - start

rows = []
for i in range(2000):
    random_seconds = random.randint(0, int(delta.total_seconds()))
    timestamp = start + timedelta(seconds=random_seconds)
    bill = round(random.uniform(10, 60), 2)
    tip = round(bill * random.uniform(0.10, 0.25), 2)
    rows.append({
        "timestamp": timestamp,
        "total_bill": bill,
        "tip": tip,
        "guests": random.randint(1, 6),
    })

df = pd.DataFrame(rows)
print(df.head(3))
print(f"\nРозмір: {len(df)} рядків, тип timestamp: {df['timestamp'].dtype}")
assert len(df) == 2000

### 9. `.dt` аксесор — datetime для цілої колонки

Коли колонка має тип `datetime64`, pandas дає `.dt` — доступ до будь-якого атрибута datetime **одразу для всієї колонки**.

In [ ]:
sample = df.head(3)["timestamp"]

print(".dt.year:   ", sample.dt.year.values)
print(".dt.month:  ", sample.dt.month.values)
print(".dt.weekday:", sample.dt.weekday.values, " (0=Пн)")
print(".dt.day_name():", sample.dt.day_name().values)

### 10. Фільтрація по року

Фільтр DataFrame — це маска `True`/`False` для кожного рядка: `df[маска]` лишає тільки рядки, де маска `True`.

In [ ]:
mask_2023 = df["timestamp"].dt.year == 2023
mask_2024 = df["timestamp"].dt.year == 2024

df_2023 = df[mask_2023]
df_2024 = df[mask_2024]

print(f"Рядків 2023: {len(df_2023)}, 2024: {len(df_2024)}, разом: {len(df_2023) + len(df_2024)}")
assert len(df_2023) + len(df_2024) == len(df)

print(f"Середній чек 2023: ${df_2023['total_bill'].mean():.2f}")
print(f"Середній чек 2024: ${df_2024['total_bill'].mean():.2f}")

### 11. Комбінована фільтрація: `&` (AND), `|` (OR), `~` (NOT)

Кожна умова — в окремих дужках, інакше пріоритет операторів зламає вираз.

In [ ]:
mask_dec_2024 = (df["timestamp"].dt.year == 2024) & (df["timestamp"].dt.month == 12)
df_dec_2024 = df[mask_dec_2024]
print(f"Грудень 2024: {len(df_dec_2024)} замовлень")

years = df_dec_2024["timestamp"].dt.year.unique()
months = df_dec_2024["timestamp"].dt.month.unique()
assert list(years) == [2024] and list(months) == [12]

mask_evening = df["timestamp"].dt.hour >= 18
df_evening = df[mask_evening]
print(f"Вечірні замовлення (після 18:00): {len(df_evening)}, середній чек ${df_evening['total_bill'].mean():.2f}")

### 12. `calendar` — ще один стандартний модуль

`calendar` не потребує встановлення й дає назви місяців/днів тижня. **Важливо:** `month_name[0] == ""` — нумерація з 1, не з 0.

In [ ]:
import calendar

print("calendar.month_name[7]  =", calendar.month_name[7])
print("calendar.day_name[0]    =", calendar.day_name[0], " (0=Понеділок)")
print("calendar.month_abbr[7]  =", calendar.month_abbr[7])
assert calendar.month_name[0] == ""

In [ ]:
print(f"{'Місяць':<12} {'Замовлень':>10} {'Виручка':>12}")
for month_num in range(1, 13):
    subset = df[df["timestamp"].dt.month == month_num]
    if len(subset) == 0:
        continue
    revenue = subset["total_bill"].sum()
    name = calendar.month_name[month_num]
    print(f"{name:<12} {len(subset):>10} ${revenue:>10,.2f}")

### 13. `Counter` — підрахунок із `collections`

`Counter` рахує, скільки разів зустрічається кожне значення — простіше за ручний `d.get(key, 0) + 1` з Уроку 6, коли треба саме порахувати входження.

In [ ]:
from collections import Counter

years_list = df["timestamp"].dt.year.tolist()
year_counter = Counter(years_list)
print("Counter по роках:", year_counter)

months_list = df["timestamp"].dt.month.tolist()
month_counter = Counter(months_list)
print("\nТоп-3 найзавантаженіші місяці:")
for month_num, count in month_counter.most_common(3):
    print(f"  {calendar.month_name[month_num]}: {count} замовлень")

assert sum(year_counter.values()) == len(df)

### 14. Похідні колонки: рахуємо один раз, фільтруємо багато разів

Замість того щоб фільтрувати по `.dt.year`/`.dt.month` щоразу наново — додаємо колонки один раз:

In [ ]:
df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["weekday"] = df["timestamp"].dt.weekday
df["hour"] = df["timestamp"].dt.hour


def meal_type(hour):
    if 11 <= hour <= 15:
        return "Lunch"
    if 17 <= hour <= 23:
        return "Dinner"
    return "Other"


df["meal"] = df["hour"].apply(meal_type)

print(df[["timestamp", "year", "month", "weekday", "meal"]].head(5))

weekend_dinner_2023 = df[(df["year"] == 2023) & (df["weekday"] >= 5) & (df["meal"] == "Dinner")]
print(f"\nВечеря у вихідні 2023: {len(weekend_dinner_2023)} замовлень")

### 15. `locale` — назви місяців/днів іншою мовою

`calendar`/`strftime("%A")` за замовчуванням дають англійські назви. `locale.setlocale()` перемикає мову — **але доступні локалі залежать від конкретної машини**: на сервері викладача чи в CI українська локаль може бути не встановлена. Тому це обов'язково обгортаємо в `try/except` — коли локаль недоступна, код повертається до англійських назв замість падіння.

In [ ]:
import locale

original_locale = locale.getlocale(locale.LC_TIME)
locale_switched = False

for candidate in ("uk_UA.UTF-8", "Ukrainian_Ukraine", "uk_UA"):
    try:
        locale.setlocale(locale.LC_TIME, candidate)
        locale_switched = True
        print(f"Локаль '{candidate}' встановлена успішно")
        break
    except locale.Error:
        continue

if not locale_switched:
    print("Українська локаль не встановлена на цій машині — лишаємось на англійських назвах")
    print("(це нормально: доступність locale залежить від ОС, не від коду)")

now = datetime.now()
print(f"\nnow.strftime('%A') = {now.strftime('%A')}")
print(f"now.strftime('%B') = {now.strftime('%B')}")

locale.setlocale(locale.LC_TIME, original_locale or "C")  # повертаємо як було

### 16. Стандартний модуль керує бізнес-рішенням

`datetime`/`calendar` тут не самоціль — вони живлять реальну класифікацію: за годиною визначаємо тип прийому їжі, за міткою часу — день тижня. Це та сама ідея, що й `meal` вище, але як окремі, перевикористовувані функції — саме такий вигляд матиме код, коли Урок [іteratory/generators] зіпре їх у більший конвеєр:

In [ ]:
def meal_type_from_hour(hour: int) -> str:
    if 11 <= hour <= 15:
        return "Lunch"
    if 17 <= hour <= 23:
        return "Dinner"
    return "Other"


def day_from_timestamp(ts) -> str:
    return ts.strftime("%A")


# Перевірка на межах діапазонів — саме тут ховаються помилки "на одиницю"
assert meal_type_from_hour(11) == "Lunch"
assert meal_type_from_hour(15) == "Lunch"
assert meal_type_from_hour(16) == "Other"
assert meal_type_from_hour(17) == "Dinner"
assert meal_type_from_hour(23) == "Dinner"
assert meal_type_from_hour(9) == "Other"

sample_ts = datetime(2024, 7, 20, 19, 45)
print(f"Приклад: {sample_ts} -> {meal_type_from_hour(sample_ts.hour)}, {day_from_timestamp(sample_ts)}")
print("Усі межові перевірки пройшли")

## 🔄 TRANSFER — самостійні задачі

Ті самі інструменти (`datetime`, `.dt`, `calendar`, `Counter`, фільтрація) — нові запитання власника ресторану.

In [ ]:
# Задача 1 — місяць із найвищим середнім чеком у 2023 році
# YOUR CODE HERE
# BEGIN SOLUTION
df_2023 = df[df["year"] == 2023]
avg_by_month = df_2023.groupby("month")["total_bill"].mean()
best_month_num = avg_by_month.idxmax()
best_month_name = calendar.month_name[best_month_num]
# END SOLUTION

print(f"Найкращий місяць 2023: {best_month_name} (avg ${avg_by_month[best_month_num]:.2f})")
assert best_month_name in list(calendar.month_name)

In [ ]:
# Задача 2 — скільки замовлень у кожному кварталі (Q1..Q4), через Counter
# YOUR CODE HERE
# BEGIN SOLUTION
def quarter_of(month):
    return f"Q{(month - 1) // 3 + 1}"


quarters = df["month"].apply(quarter_of).tolist()
quarter_counter = Counter(quarters)
# END SOLUTION

for q in ["Q1", "Q2", "Q3", "Q4"]:
    print(f"  {q}: {quarter_counter[q]} замовлень")
assert sum(quarter_counter.values()) == len(df)

In [ ]:
# Задача 3 — топ-3 години дня за виручкою
# YOUR CODE HERE
# BEGIN SOLUTION
revenue_by_hour = df.groupby("hour")["total_bill"].sum().sort_values(ascending=False)
top3_hours = revenue_by_hour.head(3)
# END SOLUTION

for hour, revenue in top3_hours.items():
    print(f"  {hour:02d}:00 -> ${revenue:.2f}")
assert len(top3_hours) == 3

In [ ]:
# Задача 4 — середній чек у вихідні vs будні, окремо для кожного року
# YOUR CODE HERE
# BEGIN SOLUTION
df["is_weekend"] = df["weekday"] >= 5
result = df.groupby(["year", "is_weekend"])["total_bill"].mean()
# END SOLUTION

for (year, is_weekend), avg in result.items():
    label = "вихідні" if is_weekend else "будні"
    print(f"  {year} {label}: ${avg:.2f}")
assert len(result) == 4  # 2 роки x 2 категорії

## ✅ Самоперевірка (5 запитань)

**1.** Чому `if __name__ == "__main__":` не виконується при `import`?

<details><summary>Відповідь</summary><code>__name__</code> дорівнює <code>'__main__'</code> тільки коли файл запущено напряму; при <code>import</code> Python встановлює <code>__name__</code> у назву модуля — умова стає хибною.</details>

**2.** Що станеться, якщо назвати свій файл `random.py` у тому самому проєкті, де є `import random`?

<details><summary>Відповідь</summary>Python знайде твій файл раніше за стандартний модуль (shadowing) — <code>import random</code> підвантажить твій код, а не стандартну бібліотеку, і всі функції звідти (<code>random.randint</code> тощо) стануть недоступні або зламаються.</details>

**3.** Чому `locale.setlocale()` обов'язково обгортати в `try/except` у навчальному коді?

<details><summary>Відповідь</summary>Доступні локалі залежать від конкретної машини (ОС, встановлені пакети) — код, що працює на одному комп'ютері, може впасти на іншому лише через відсутність локалі, хоча логіка коду правильна.</details>

**4.** У чому різниця між `df["timestamp"].dt.year` і `dt.year` для одного об'єкта `datetime`?

<details><summary>Відповідь</summary><code>dt.year</code> — атрибут одного об'єкта <code>datetime</code>, повертає одне число. <code>df["timestamp"].dt.year</code> — <code>.dt</code>-аксесор pandas, застосовує той самий атрибут одразу до кожного рядка колонки й повертає <code>Series</code>.</details>

**5.** `calendar.month_name[0]` — чому це порожній рядок, а не `"January"`?

<details><summary>Відповідь</summary>Місяці нумеруються природно, з 1 до 12 (<code>datetime.month</code> теж 1-12) — індекс 0 залишений порожнім спеціально, щоб <code>month_name[7]</code> одразу давало липень без ручного зсуву <code>-1</code>.</details>

### Шпаргалка

```python
# import
import math                  # модуль цілком
from math import pi, sqrt    # конкретні імена
from math import pi as PI    # з псевдонімом

# datetime
from datetime import datetime
dt = datetime(2024, 7, 20, 19, 45)
dt.year, dt.month, dt.day, dt.hour, dt.weekday()
dt.strftime("%Y-%m-%d")      # -> рядок

# pandas .dt
df["col"].dt.year / .dt.month / .dt.weekday / .dt.day_name()

# calendar
import calendar
calendar.month_name[7]       # "July" (нумерація з 1!)
calendar.day_name[0]         # "Monday"

# collections.Counter
from collections import Counter
Counter(список).most_common(3)

# locale (обов'язково try/except — доступність залежить від машини)
import locale
try:
    locale.setlocale(locale.LC_TIME, "uk_UA.UTF-8")
except locale.Error:
    pass
```

## Далі

Наступного разу — Урок «Ітератори й генератори»: коли даних стає багато (500 000+ записів), звичайний список і `pandas.DataFrame` перестають вміщатись у пам'ять одразу — знадобляться генератори, які обробляють дані по одному запису, не завантажуючи все відразу. Функції `meal_type_from_hour`/`day_from_timestamp` із цього уроку — ті самі, що там використовуються для трансформації `RawOrder -> Order` у пайплайні.